# Stage 5: Evaluation
Loads the saved model and artifacts from S3, scores the validation set, computes MAE, RMSE, R2 and MAPE, saves evaluation report and prediction sample.

Input: s3://ins-churn-data/model_artifacts/ and features/val.csv

Output: s3://ins-churn-data/model_output/metrics.json, predictions_sample.csv, feature_importance.csv, evaluation.json

In [ ]:
import boto3, pandas as pd, numpy as np, io, os, json, logging, pickle
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
log = logging.getLogger(__name__)

BUCKET_NAME = 'ins-churn-data'
REGION = 'us-east-1'
LOCAL_TMP = '/tmp/ins_churn'
os.makedirs(LOCAL_TMP, exist_ok=True)

s3 = boto3.client('s3', region_name=REGION)


def read_csv_from_s3(bucket, key, **kwargs):
    response = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(io.BytesIO(response['Body'].read()), **kwargs)


def write_csv_to_s3(df, bucket, key):
    buffer = io.StringIO()
    df.to_csv(buffer, index=False)
    s3.put_object(Bucket=bucket, Key=key, Body=buffer.getvalue())
    log.info('  saved to s3://%s/%s (%d rows)', bucket, key, len(df))


def write_json_to_s3(data, bucket, key):
    s3.put_object(Bucket=bucket, Key=key,
                  Body=json.dumps(data, indent=2, default=str))


log.info('Helpers ready.')

In [ ]:
# Interaction features and prepare_xy, must match the training definitions
TARGET_COL = 'customer_lifetime_value'


def add_interaction_features(df):
    out = df.copy()

    def safe_col(name):
        return out[name] if name in out.columns else pd.Series(0, index=out.index)

    out['premium_x_tenure'] = safe_col('premium_amount') * safe_col('policy_tenure_months')
    out['delay_x_complaints'] = safe_col('payment_delay_days') * safe_col('customer_complaints')
    out['claims_x_premium'] = safe_col('claim_frequency') * safe_col('premium_amount')
    out['margin_x_tenure'] = safe_col('policy_margin') * safe_col('policy_tenure_months')
    out['agent_tenure_x_premium'] = safe_col('agent_tenure_days') * safe_col('premium_amount')
    out['cust_tenure_x_margin_pct'] = safe_col('customer_tenure_days') * safe_col('margin_pct')
    return out


def prepare_xy(df, feature_names=None, scaler=None):
    if feature_names is not None:
        cols = feature_names + ([TARGET_COL] if TARGET_COL in df.columns else [])
        df = df.reindex(columns=cols, fill_value=0)
    y_raw = df[TARGET_COL].values if TARGET_COL in df.columns else None
    y_log = np.log1p(y_raw) if y_raw is not None else None
    feat_cols = [c for c in df.columns if c != TARGET_COL]
    X = df[feat_cols].values.astype(np.float64)
    X_scaled = scaler.transform(X) if scaler is not None else X
    return X_scaled, feat_cols, y_raw, y_log, scaler


log.info('Helper functions defined.')

In [ ]:
# Load model artifacts from S3
xgb_tmp = os.path.join(LOCAL_TMP, 'xgb_model_eval.json')
s3.download_file(BUCKET_NAME, 'model_artifacts/xgb_model.json', xgb_tmp)
model = xgb.XGBRegressor()
model.load_model(xgb_tmp)

scaler_eval = pickle.loads(
    s3.get_object(Bucket=BUCKET_NAME, Key='model_artifacts/scaler.pkl')['Body'].read()
)
feat_eval = json.loads(
    s3.get_object(Bucket=BUCKET_NAME, Key='model_artifacts/feature_names.json')['Body'].read()
)
model_config = json.loads(
    s3.get_object(Bucket=BUCKET_NAME, Key='model_artifacts/model_config.json')['Body'].read()
)

log.info('Model loaded. Best iteration: %s, Features: %d',
         model_config.get('best_iteration', 'N/A'), len(feat_eval))

In [ ]:
# Score validation set
val = read_csv_from_s3(BUCKET_NAME, 'features/val.csv')
val_e = add_interaction_features(val)
X_ev, _, y_ev_raw, _, _ = prepare_xy(val_e, feature_names=feat_eval, scaler=scaler_eval)

preds_ev = np.expm1(model.predict(X_ev))   # reverse log transform
residuals = y_ev_raw - preds_ev

mae = mean_absolute_error(y_ev_raw, preds_ev)
rmse = np.sqrt(mean_squared_error(y_ev_raw, preds_ev))
r2 = r2_score(y_ev_raw, preds_ev)
mape = np.mean(np.abs(residuals / np.clip(y_ev_raw, 1, None))) * 100

# Naive baseline (mean prediction)
baseline_rmse = np.sqrt(mean_squared_error(y_ev_raw, np.full_like(y_ev_raw, y_ev_raw.mean())))
beats_baseline = rmse < baseline_rmse

log.info('R2=%.4f  RMSE=%.2f  MAE=%.2f  MAPE=%.2f%%', r2, rmse, mae, mape)
log.info('Baseline RMSE=%.2f, beats baseline: %s', baseline_rmse, beats_baseline)
if r2 >= 0.70 and rmse <= 5000:
    log.info('Gate: PASS')
else:
    log.info('Gate: FAIL')

In [ ]:
# Save outputs to S3

# Metrics
write_json_to_s3(
    {'R2': round(r2, 4), 'RMSE': round(rmse, 2), 'MAE': round(mae, 2), 'MAPE': round(mape, 2)},
    BUCKET_NAME, 'model_output/metrics.json'
)

# Evaluation summary (includes baseline)
write_json_to_s3(
    {'R2': round(r2, 4), 'RMSE': round(rmse, 2), 'MAE': round(mae, 2),
     'MAPE': round(mape, 2), 'baseline_rmse': round(baseline_rmse, 2),
     'beats_baseline': bool(beats_baseline),
     'gate_pass': bool(r2 >= 0.70 and rmse <= 5000)},
    BUCKET_NAME, 'model_output/evaluation.json'
)

# Feature importance
importance = pd.DataFrame({'feature': feat_eval, 'importance': model.feature_importances_})
importance = importance.sort_values('importance', ascending=False)
write_csv_to_s3(importance, BUCKET_NAME, 'model_output/feature_importance.csv')
display(importance.head(10))

# Prediction sample (first 200 rows)
sample = val_e.head(200).copy().reindex(columns=feat_eval, fill_value=0)
sample['actual_clv'] = y_ev_raw[:200]
sample['predicted_clv'] = preds_ev[:200].round(2)
sample['residual'] = residuals[:200].round(2)
write_csv_to_s3(sample, BUCKET_NAME, 'model_output/predictions_sample.csv')

log.info('Evaluation saved to S3.')